# Two Follow-Ups: an Independent Foldability Direction, and the Fold-Failure Fix

Bundled into one notebook because both are ProtGPT2-layer-12-only, both are cheap, and combining
them means one model load and one ESMFold load instead of two separate sessions.

## Fix 1 — an actually-independent foldability direction

`36-ai4dd-foldability-direction-steering.ipynb` found that its foldability vector and the
repetition vector built from the same pool have cosine similarity **+0.98** — essentially the same
direction (`notes/locked-results.md` §1m). That means `36`'s result is not an independent test of
"does steering toward the desired property help" — it's close to a re-measurement of the
repetition-vector result under a different label.

This notebook builds the genuine independent test the header of `36` motivated but didn't build:
**orthogonalize the foldability vector against the repetition vector** (project out the shared
component, the same technique `35`'s `ORTHOGONAL` condition used against the naive vector), and
steer with the residual — the part of "well-folding vs. badly-folding" that repetition doesn't
already explain. Tested at the same gentle-weighted ladder as `36` (0.25x/0.5x/1x/2x), for the same
reason: any real benefit, if it exists, lives below the saturating regime.

## Fix 2 — refolding `ORTHOGONAL_2x` with a length-aware evaluator

`35-ai4dd-random-direction-control.ipynb`'s `ORTHOGONAL_2x` condition reported 2.0% collapse, but
that number is an artifact: reconstructing from the notebook's own filtered/unfiltered pLDDT means
(`notes/locked-results.md` §1l), ~49 of 50 sequences never actually finished folding, almost
certainly because that condition's generations were unusually long (mean 268.6 residues vs.
≤198 for every other condition in that run) and ESMFold ran out of memory. The project's
collapse metric, `int(0.0 < plddt < 60.0)`, silently counts a total fold failure as a success.

This notebook rebuilds the exact same `ORTHOGONAL` direction (a random vector with the naive
`v_L` component projected out, same construction as `35`) and folds it with a
**length-aware evaluator**: cap the cleaned sequence to a safe maximum length before folding, retry
once at half that length if ESMFold still throws, and explicitly report the fold-success count
instead of silently absorbing failures into the collapse label. This same evaluator is used for
every condition in this notebook, including Fix 1's, so the fold-failure gap found in `35` doesn't
propagate here.

Kaggle setup: Accelerator = **GPU T4 x1 or x2**, Internet = **ON**. Expect ~2-2.5 hours
(200 candidate folds + ~300 condition folds).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding

torch.manual_seed(2024)
np.random.seed(2024)

device = "cuda" if torch.cuda.is_available() else "cpu"
REFERENCE_NORM = 583.998
TARGET_LAYER = 12
N_CANDIDATES = 200
N_PER_CONDITION = 50

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

print("Setup complete. CUDA available:", torch.cuda.is_available())


Setup complete. CUDA available: True


In [2]:
# --- Scoring functions, unchanged from 24/26/36. ---
ALPHABET_SIZE = 20

def h_norm(seq):
    if not seq:
        return 0.0
    counts = collections.Counter(seq)
    total = len(seq)
    ent = -sum((c / total) * math.log2(c / total) for c in counts.values())
    return ent / math.log2(ALPHABET_SIZE)

def distinct_n(seq, n):
    if len(seq) < n:
        return 1.0
    grams = [seq[i:i + n] for i in range(len(seq) - n + 1)]
    return len(set(grams)) / len(grams)

def homopolymer_runs(seq):
    if not seq:
        return []
    runs, run_len = [], 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return runs

def r_hpoly(seq, k=4):
    if not seq:
        return 1.0
    T = len(seq)
    penalty = sum(l for l in homopolymer_runs(seq) if l >= k)
    return max(0.0, 1.0 - penalty / T)

def repetition_score(seq):
    return float(np.mean([h_norm(seq), distinct_n(seq, 2), distinct_n(seq, 3), r_hpoly(seq)]))

def utility_score(plddt, ptm):
    return float(np.mean([plddt / 100.0, ptm]))

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

# --- THE FIX: a length-aware evaluator. Every earlier notebook's fold_one caps nothing and
#     retries nothing -- a sequence long enough to OOM ESMFold silently returns (0.0, 0.0),
#     which the project's collapse label then counts as a SUCCESS (0.0 < plddt < 60.0 is False
#     when plddt==0.0). This is exactly the gap 35's ORTHOGONAL_2x condition exposed. ---

MAX_FOLD_LEN = 300   # generous -- covers every condition's mean length seen in this project so
                     # far except 35's ORTHOGONAL_2x outlier tail; chosen to avoid truncating
                     # ordinary sequences while still bounding ESMFold's memory cost.

class SafeStructuralEvaluator:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def _fold_raw(self, cleaned):
        inputs = self.tokenizer([cleaned], return_tensors="pt", add_special_tokens=False).to(self.device)
        with torch.no_grad():
            out = self.model(**inputs)
        raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
        plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
        ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
        return plddt, ptm

    def fold_one(self, seq, max_len=MAX_FOLD_LEN):
        # Returns (plddt, ptm, fold_ok, truncated_to). fold_ok is False only if folding
        # genuinely could not be completed even after truncation+retry -- distinct from a
        # collapsed-but-successfully-folded sequence.
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0, False, None

        truncated_to = None
        working = cleaned
        if len(working) > max_len:
            working = working[:max_len]
            truncated_to = max_len

        try:
            plddt, ptm = self._fold_raw(working)
            return plddt, ptm, True, truncated_to
        except RuntimeError:
            clear_gpu()

        # Retry once at half length -- covers the case where even MAX_FOLD_LEN is too long for
        # whatever GPU memory happens to be free at this point in the session.
        half_len = max(10, len(working) // 2)
        if half_len < len(working):
            try:
                plddt, ptm = self._fold_raw(working[:half_len])
                return plddt, ptm, True, half_len
            except RuntimeError:
                clear_gpu()

        return 0.0, 0.0, False, truncated_to

def fold_records_safe(records, evaluator, key="sequence"):
    for r in records:
        plddt, ptm, fold_ok, truncated_to = evaluator.fold_one(r[key])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["fold_ok"] = fold_ok
        r["truncated_to"] = truncated_to
        r["collapse"] = int(0.0 < plddt < 60.0)
    return records

print(f"Safe evaluator ready. MAX_FOLD_LEN={MAX_FOLD_LEN}, retries once at half-length on failure.")


Safe evaluator ready. MAX_FOLD_LEN=300, retries once at half-length on failure.


In [3]:
# --- Candidate pool: same construction as 24/36. ---
UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

FALLBACK_SEQS = [
    "NLYIQWLKDGGPSSGRPPPS",
    "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF",
    "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

USED_FALLBACK = False
if not reference_seqs:
    USED_FALLBACK = True
    print("!" * 78)
    print("!! UniProt fetch returned NOTHING -- Kaggle's Internet toggle is likely OFF.")
    print("!! Fix: Notebook sidebar -> Session options -> Internet -> ON, then re-run.")
    print("!" * 78)
    reference_seqs = [(f"local{i + 1}", s) for i, s in enumerate(FALLBACK_SEQS)]

def build_prefix_pool(reference_seqs, n_prefixes, min_len=10, max_len=15, seed=11):
    if not reference_seqs:
        raise RuntimeError("No reference sequences available -- check Internet settings.")
    rng = np.random.RandomState(seed)
    prefixes = []
    for i in range(n_prefixes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        plen = rng.randint(min_len, max_len + 1)
        start = rng.randint(0, max(1, len(seq) - plen))
        prefixes.append(seq[start:start + plen])
    return prefixes

candidate_prefixes = build_prefix_pool(reference_seqs, n_prefixes=N_CANDIDATES)
print(f"Built {len(candidate_prefixes)} candidate prefixes.")

print(f"Loading ProtGPT2 on {device}...")
tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
plm_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
plm_model.eval()

def generate_natural(tokenizer, model, prompts, max_len=50, seed=0):
    torch.manual_seed(seed)
    records = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, pad_token_id=tokenizer.eos_token_id
            )
        seq = tokenizer.decode(output_ids[0], skip_special_tokens=True).replace(" ", "")
        gen_only = seq[len(prompt):] if seq.startswith(prompt) else seq
        records.append({"prompt": prompt, "sequence": seq, "gen_only": gen_only})
    clear_gpu()
    return records

print(f"=== Generating N={N_CANDIDATES} natural candidate sequences ===")
candidate_records = generate_natural(tokenizer, plm_model, candidate_prefixes, max_len=50, seed=505)

print("=== Freeing ProtGPT2 while ESMFold folds the pool ===")
del plm_model
clear_gpu()

evaluator = SafeStructuralEvaluator()
print("Folding and scoring the candidate pool...")
for r in candidate_records:
    r["repetition_score"] = repetition_score(r["gen_only"])
candidate_records = fold_records_safe(candidate_records, evaluator, key="sequence")
for r in candidate_records:
    r["utility_score"] = utility_score(r["plddt"], r["ptm"]) if r["plddt"] > 0 else 0.0
del evaluator
clear_gpu()

n_fold_ok = sum(1 for r in candidate_records if r["fold_ok"])
valid_candidates = [r for r in candidate_records if r["plddt"] > 0.0]
print(f"\n{n_fold_ok}/{len(candidate_records)} candidates folded successfully "
      f"({sum(1 for r in candidate_records if r['truncated_to'])} needed truncation).")
print(f"Mean pLDDT: {np.mean([r['plddt'] for r in valid_candidates]):.2f}, "
      f"natural collapse rate: {np.mean([r['collapse'] for r in valid_candidates]):.1%}")


Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  fetched P42212: 238 residues
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  fetched P00648: 157 residues
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues


  fetched P00441: 154 residues
Built 200 candidate prefixes.
Loading ProtGPT2 on cuda...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.bias   | MISSING    | 
esm.contact_head.regression.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding and scoring the candidate pool...

200/200 candidates folded successfully (0 needed truncation).
Mean pLDDT: 57.43, natural collapse rate: 57.0%


In [4]:
# --- Build both vectors from the same pool: repetition (24's recipe) and foldability (36's
#     recipe), then orthogonalize foldability against repetition -- THE actual new construction
#     this notebook exists to test. ---

def utility_match(pool_a, pool_b, key, tolerance, max_iters=200):
    a, b = list(pool_a), list(pool_b)
    for _ in range(max_iters):
        mean_a = np.mean([r[key] for r in a])
        mean_b = np.mean([r[key] for r in b])
        gap = mean_a - mean_b
        if abs(gap) <= tolerance or min(len(a), len(b)) <= 15:
            break
        if gap > 0:
            a.sort(key=lambda r: -r[key])
            a.pop(0)
        else:
            b.sort(key=lambda r: r[key])
            b.pop(0)
    return a, b

QUANTILE = 0.30

# Repetition split (24's convention): D+/D- by repetition extremes, matched on utility.
sorted_by_rep = sorted(valid_candidates, key=lambda r: r["repetition_score"])
n_side = max(10, int(len(sorted_by_rep) * QUANTILE))
rep_dplus_raw = sorted_by_rep[-n_side:]
rep_dminus_raw = sorted_by_rep[:n_side]
rep_dplus, rep_dminus = utility_match(rep_dplus_raw, rep_dminus_raw, "utility_score", 0.05)
print(f"Repetition split: D+ n={len(rep_dplus)}, D- n={len(rep_dminus)}, "
      f"utility gap {abs(np.mean([r['utility_score'] for r in rep_dplus]) - np.mean([r['utility_score'] for r in rep_dminus])):.3f}")

# Foldability split (36's convention): D+/D- by utility extremes, matched on repetition.
sorted_by_util = sorted(valid_candidates, key=lambda r: r["utility_score"])
fold_dhigh_raw = sorted_by_util[-n_side:]
fold_dlow_raw = sorted_by_util[:n_side]
fold_dhigh, fold_dlow = utility_match(fold_dhigh_raw, fold_dlow_raw, "repetition_score", 0.05)
print(f"Foldability split: D+ n={len(fold_dhigh)}, D- n={len(fold_dlow)}, "
      f"repetition gap {abs(np.mean([r['repetition_score'] for r in fold_dhigh]) - np.mean([r['repetition_score'] for r in fold_dlow])):.3f}")

print(f"\nReloading ProtGPT2 on {device} to extract activations...")
plm_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
plm_model.eval()

def get_mean_activation(model, tokenizer, seq_list, layer):
    acts = []
    for seq in seq_list:
        inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=256).to(device)
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
            acts.append(out.hidden_states[layer].mean(dim=1).squeeze(0).cpu())
    return torch.stack(acts)

pos_r = get_mean_activation(plm_model, tokenizer, [r["sequence"] for r in rep_dplus], TARGET_LAYER)
neg_r = get_mean_activation(plm_model, tokenizer, [r["sequence"] for r in rep_dminus], TARGET_LAYER)
v_rep_raw = pos_r.mean(dim=0) - neg_r.mean(dim=0)
v_rep = v_rep_raw * (REFERENCE_NORM / v_rep_raw.norm().item())

pos_f = get_mean_activation(plm_model, tokenizer, [r["sequence"] for r in fold_dhigh], TARGET_LAYER)
neg_f = get_mean_activation(plm_model, tokenizer, [r["sequence"] for r in fold_dlow], TARGET_LAYER)
v_fold_raw = pos_f.mean(dim=0) - neg_f.mean(dim=0)

def cos(a, b):
    a, b = a.float(), b.float()
    return float(torch.dot(a, b) / (a.norm() * b.norm()))

cos_before = cos(v_fold_raw, v_rep_raw)
print(f"\nCosine(foldability_raw, repetition_raw) BEFORE orthogonalization: {cos_before:+.4f}")
print(f"(36 found +0.98 on its own independently-built pool -- a similar value here would")
print(f" confirm that finding rather than being a fluke of that specific run.)")

# --- THE new construction: project out the repetition component from the foldability vector. ---
v_rep_unit = v_rep_raw / v_rep_raw.norm()
v_fold_orth_raw = v_fold_raw - torch.dot(v_fold_raw, v_rep_unit) * v_rep_unit
cos_after = cos(v_fold_orth_raw, v_rep_raw)
print(f"Cosine(foldability_orthogonalized, repetition_raw) AFTER: {cos_after:+.4f}  (expect ~0)")

v_fold_orth = (v_fold_orth_raw * (REFERENCE_NORM / v_fold_orth_raw.norm().item())).to(device)
v_rep_dev = v_rep.to(device)
print(f"\nv_fold_orth norm after rescaling: {v_fold_orth.norm().item():.4f}")


Repetition split: D+ n=31, D- n=60, utility gap 0.047
Foldability split: D+ n=18, D- n=60, repetition gap 0.046

Reloading ProtGPT2 on cuda to extract activations...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Cosine(foldability_raw, repetition_raw) BEFORE orthogonalization: +0.9819
(36 found +0.98 on its own independently-built pool -- a similar value here would
 confirm that finding rather than being a fluke of that specific run.)
Cosine(foldability_orthogonalized, repetition_raw) AFTER: -0.0000  (expect ~0)

v_fold_orth norm after rescaling: 583.9980


In [5]:
# --- Rebuild the naive v_L and the ORTHOGONAL direction exactly as in 35, for the refold fix.
#     Reusing 35's exact construction (not this notebook's pool-based vectors) because the thing
#     being fixed is specifically 35's ORTHOGONAL_2x condition -- the fix should test the SAME
#     direction, just fold it properly this time. ---

positive_seqs = [
    "NLYIQWLKDGGPSSGRPPPS", "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF", "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]
degenerate_seqs = [
    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA", "LGLGLGLGLGLGLGLGLGLGLGLGLGLGLGLGLGL",
    "GGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGG", "SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS",
    "PGPGPGPGPGPGPGPGPGPGPGPGPGPGPGPGPGP", "QWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQ",
]
pos_acts_naive = get_mean_activation(plm_model, tokenizer, positive_seqs, TARGET_LAYER)
neg_acts_naive = get_mean_activation(plm_model, tokenizer, degenerate_seqs, TARGET_LAYER)
v_L_raw = pos_acts_naive.mean(dim=0) - neg_acts_naive.mean(dim=0)
v_L = (v_L_raw * (REFERENCE_NORM / v_L_raw.norm().item()))

gen = torch.Generator(device="cpu").manual_seed(31337)
g = torch.randn(v_L.shape[0], generator=gen)
v_L_cpu = v_L.detach().cpu()
g_orth = g - (torch.dot(g, v_L_cpu) / torch.dot(v_L_cpu, v_L_cpu)) * v_L_cpu
v_orthogonal = (g_orth * (REFERENCE_NORM / g_orth.norm())).to(device)

print(f"Rebuilt naive v_L (norm {v_L.norm().item():.4f}) and ORTHOGONAL direction "
      f"(norm {v_orthogonal.norm().item():.4f}, cosine to v_L: {cos(v_orthogonal.cpu(), v_L_cpu):+.4f}) "
      f"-- same construction as 35.")


Rebuilt naive v_L (norm 583.9980) and ORTHOGONAL direction (norm 583.9980, cosine to v_L: -0.0000) -- same construction as 35.


In [6]:
# --- Generate every condition. Fix 1's ladder + Fix 2's single refold condition, in one pass. ---

def generate_with_vector_steering(model, tokenizer, target_layer, steering_vector, prompts,
                                  max_len=50, seed=0):
    torch.manual_seed(seed)
    model.eval()
    v = None if steering_vector is None else steering_vector.to(device)

    def hook(module, inp, out):
        if v is None:
            return out
        return (out[0] + v,)

    records = []
    for prompt in prompts:
        handle = model.transformer.h[target_layer].register_forward_hook(hook)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, pad_token_id=tokenizer.eos_token_id
            )
        handle.remove()
        seq = tokenizer.decode(output_ids[0], skip_special_tokens=True).replace(" ", "")
        gen_part = seq[len(prompt):] if seq.startswith(prompt) else seq
        records.append({"prompt": prompt, "sequence": seq, "gen_only": gen_part,
                        "entropy": calculate_entropy(gen_part)})
    clear_gpu()
    return records

steer_prefixes = build_prefix_pool(reference_seqs, n_prefixes=300, seed=23)

CONDITION_SPECS = [
    ("CONTROL",           None,                     111),
    ("FOLD_ORTH_0.25x",   v_fold_orth * 0.25,       222),
    ("FOLD_ORTH_0.5x",    v_fold_orth * 0.5,        333),
    ("FOLD_ORTH_1x",      v_fold_orth * 1.0,        444),
    ("FOLD_ORTH_2x",      v_fold_orth * 2.0,        555),
    ("ORTHOGONAL_REFOLD_2x", v_orthogonal * 2.0,    777),   # same seed as 35's ORTHOGONAL_2x
]

conditions = {}
for i, (name, vec, seed) in enumerate(CONDITION_SPECS):
    lo, hi = i * N_PER_CONDITION, (i + 1) * N_PER_CONDITION
    print(f"=== {name} (prefixes {lo}:{hi}, seed {seed}) ===")
    conditions[name] = generate_with_vector_steering(
        plm_model, tokenizer, TARGET_LAYER, vec, steer_prefixes[lo:hi], seed=seed)

print("\n=== Freeing ProtGPT2 from GPU ===")
del plm_model
clear_gpu()


=== CONTROL (prefixes 0:50, seed 111) ===
=== FOLD_ORTH_0.25x (prefixes 50:100, seed 222) ===
=== FOLD_ORTH_0.5x (prefixes 100:150, seed 333) ===
=== FOLD_ORTH_1x (prefixes 150:200, seed 444) ===
=== FOLD_ORTH_2x (prefixes 200:250, seed 555) ===
=== ORTHOGONAL_REFOLD_2x (prefixes 250:300, seed 777) ===

=== Freeing ProtGPT2 from GPU ===


In [7]:
# --- Fold everything with the SAFE evaluator, tracking fold-success explicitly. ---

evaluator2 = SafeStructuralEvaluator()
for name in conditions:
    print(f"Folding {name}...")
    conditions[name] = fold_records_safe(conditions[name], evaluator2, key="sequence")
del evaluator2
clear_gpu()

from scipy.stats import fisher_exact

def wilson_ci(k, n, z=1.959963985):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = (z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / d
    return (max(0.0, centre - half), min(1.0, centre + half))

print(f"\n{'Condition':22s} {'N':>4s} {'FoldOK':>7s} {'Trunc':>6s} {'Entropy':>9s} {'pLDDT':>8s} {'Collapse%':>10s}")
print("-" * 78)
summary = {}
for name, recs in conditions.items():
    n = len(recs)
    n_ok = sum(1 for r in recs if r["fold_ok"])
    n_trunc = sum(1 for r in recs if r["truncated_to"])
    k = int(np.sum([r["collapse"] for r in recs]))
    plddts = [r["plddt"] for r in recs if r["plddt"] > 0]
    summary[name] = {"k": k, "n": n, "rate": k / n, "ci": wilson_ci(k, n), "fold_ok": n_ok}
    print(f"{name:22s} {n:4d} {n_ok:3d}/{n:<3d} {n_trunc:6d} "
          f"{np.mean([r['entropy'] for r in recs]):9.3f} "
          f"{(np.mean(plddts) if plddts else 0.0):8.2f} {k / n * 100:9.1f}%")


Loading ESMFold...


Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.bias   | MISSING    | 
esm.contact_head.regression.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding CONTROL...
Folding FOLD_ORTH_0.25x...
Folding FOLD_ORTH_0.5x...
Folding FOLD_ORTH_1x...
Folding FOLD_ORTH_2x...
Folding ORTHOGONAL_REFOLD_2x...

Condition                 N  FoldOK  Trunc   Entropy    pLDDT  Collapse%
------------------------------------------------------------------------------
CONTROL                  50  50/50       0     2.847    59.17      52.0%
FOLD_ORTH_0.25x          50  50/50       0     2.140    58.26      60.0%
FOLD_ORTH_0.5x           50  50/50       0     1.844    55.54      70.0%
FOLD_ORTH_1x             50  50/50       0     2.881    50.90      88.0%
FOLD_ORTH_2x             50  50/50       0     3.875    32.67     100.0%
ORTHOGONAL_REFOLD_2x     50  50/50       0     4.082    31.22     100.0%


In [8]:
# --- Analysis for both fixes. ---
ctrl = summary["CONTROL"]

print("=" * 96)
print("FIX 1 -- does an orthogonalized foldability direction do anything a plain one doesn't?")
print("=" * 96)
print(f"{'Condition':18s} {'rate':>8s} {'95% CI':>20s} {'vs CONTROL p':>13s}")
print("-" * 96)
for name in ["CONTROL", "FOLD_ORTH_0.25x", "FOLD_ORTH_0.5x", "FOLD_ORTH_1x", "FOLD_ORTH_2x"]:
    s = summary[name]
    ci_str = "[{:.1%}, {:.1%}]".format(*s["ci"])
    if name == "CONTROL":
        p_str = "--"
    else:
        _, p = fisher_exact([[s["k"], s["n"] - s["k"]], [ctrl["k"], ctrl["n"] - ctrl["k"]]])
        p_str = f"{p:.4f}"
    print(f"{name:18s} {s['rate']:7.1%} {ci_str:>20s} {p_str:>13s}")

print()
print(f"Cosine(foldability, repetition) before orthogonalization: {cos_before:+.4f}")
print(f"Cosine(foldability_orthogonalized, repetition) after:     {cos_after:+.4f}")
print()
print("Compare against 36's non-orthogonalized result (locked §1m):")
print("  CONTROL 52.0% -> FOLD_0.25x 46.0% -> FOLD_0.5x 46.0% -> FOLD_1x 62.0% -> FOLD_2x 90.0%")
best = min(("FOLD_ORTH_0.25x", "FOLD_ORTH_0.5x", "FOLD_ORTH_1x"), key=lambda n: summary[n]["rate"])
_, p_best = fisher_exact([[summary[best]["k"], summary[best]["n"] - summary[best]["k"]],
                          [ctrl["k"], ctrl["n"] - ctrl["k"]]])
if summary[best]["rate"] < ctrl["rate"] and p_best < 0.05:
    print(f"\n  ==> The orthogonalized foldability direction ({best}) shows a SIGNIFICANT")
    print(f"      improvement over CONTROL. This is the first evidence of a genuinely")
    print(f"      independent, beneficial steering direction found in this project -- a real")
    print(f"      headline result. Report cos_before/cos_after alongside it so the construction")
    print(f"      is fully transparent.")
else:
    print(f"\n  ==> No dose shows a significant improvement (best: {best} at "
          f"{summary[best]['rate']:.1%}, p={p_best:.4f}). Combined with 36's result, this is now")
    print(f"      two independent attempts (co-linear and orthogonalized) at steering toward")
    print(f"      foldability, neither of which finds a beneficial direction on ProtGPT2 layer")
    print(f"      12 -- a stronger, two-pronged version of 'even aiming at the desired property")
    print(f"      does not help', not weakened by the orthogonalization not mattering.")

print()
print("=" * 96)
print("FIX 2 -- does ORTHOGONAL_REFOLD_2x give a valid number now?")
print("=" * 96)
refold = summary["ORTHOGONAL_REFOLD_2x"]
print(f"35's original ORTHOGONAL_2x (invalid, ~49/50 fold failures): reported 2.0% collapse")
print(f"This refold, same construction, safe evaluator: "
      f"{refold['fold_ok']}/{refold['n']} folded successfully, "
      f"{refold['rate']:.1%} collapse, 95% CI [{refold['ci'][0]:.1%}, {refold['ci'][1]:.1%}]")
if refold["fold_ok"] == refold["n"]:
    print("\n  ==> Clean fold-success rate. This number is now valid -- use it in place of 35's")
    print("      ORTHOGONAL_2x wherever that condition is cited, and update locked-results.md")
    print("      §1l to replace 'INVALID' with this figure.")
else:
    print(f"\n  ==> Still {refold['n'] - refold['fold_ok']} unresolved fold failures even after the")
    print(f"      length cap and retry. Report the fold-success rate alongside any collapse")
    print(f"      number from this condition, and consider a lower MAX_FOLD_LEN if this recurs.")


FIX 1 -- does an orthogonalized foldability direction do anything a plain one doesn't?
Condition              rate               95% CI  vs CONTROL p
------------------------------------------------------------------------------------------------
CONTROL              52.0%       [38.5%, 65.2%]            --
FOLD_ORTH_0.25x      60.0%       [46.2%, 72.4%]        0.5459
FOLD_ORTH_0.5x       70.0%       [56.2%, 80.9%]        0.1004
FOLD_ORTH_1x         88.0%       [76.2%, 94.4%]        0.0002
FOLD_ORTH_2x        100.0%      [92.9%, 100.0%]        0.0000

Cosine(foldability, repetition) before orthogonalization: +0.9819
Cosine(foldability_orthogonalized, repetition) after:     -0.0000

Compare against 36's non-orthogonalized result (locked §1m):
  CONTROL 52.0% -> FOLD_0.25x 46.0% -> FOLD_0.5x 46.0% -> FOLD_1x 62.0% -> FOLD_2x 90.0%

  ==> No dose shows a significant improvement (best: FOLD_ORTH_0.25x at 60.0%, p=0.5459). Combined with 36's result, this is now
      two independent attempt

In [9]:
# --- Persist. ---
rows = []
for name, recs in conditions.items():
    for i, r in enumerate(recs):
        rows.append({
            "condition": name, "idx": i, "prompt": r["prompt"], "sequence": r["sequence"],
            "gen_only": r["gen_only"], "gen_length": len(r["gen_only"]),
            "usable_length": sum(1 for a in r["gen_only"] if a in VALID_AA),
            "entropy": r["entropy"], "plddt": r["plddt"], "ptm": r["ptm"],
            "fold_ok": r["fold_ok"], "truncated_to": r["truncated_to"], "collapse": r["collapse"],
        })
pd.DataFrame(rows).to_csv("orthogonalized_foldability_and_refold_sequences.csv", index=False)

pd.DataFrame([{
    "condition": n, "collapsed": s["k"], "n": s["n"], "collapse_rate": s["rate"],
    "ci_lo": s["ci"][0], "ci_hi": s["ci"][1], "fold_ok": s["fold_ok"],
} for n, s in summary.items()]).to_csv("orthogonalized_foldability_and_refold_summary.csv", index=False)

pd.DataFrame([{
    "cos_foldability_repetition_before": cos_before,
    "cos_foldability_repetition_after": cos_after,
    "v_fold_orth_norm": v_fold_orth.norm().item(),
}]).to_csv("orthogonalized_foldability_vectors.csv", index=False)

print("Saved:")
print("  orthogonalized_foldability_and_refold_sequences.csv")
print("  orthogonalized_foldability_and_refold_summary.csv")
print("  orthogonalized_foldability_vectors.csv")
print()
print("Update notes/locked-results.md §1m (orthogonalized-direction follow-up) and §1l")
print("(replace ORTHOGONAL_2x's invalid number with this notebook's ORTHOGONAL_REFOLD_2x).")


Saved:
  orthogonalized_foldability_and_refold_sequences.csv
  orthogonalized_foldability_and_refold_summary.csv
  orthogonalized_foldability_vectors.csv

Update notes/locked-results.md §1m (orthogonalized-direction follow-up) and §1l
(replace ORTHOGONAL_2x's invalid number with this notebook's ORTHOGONAL_REFOLD_2x).
